# Kaza — Part 5: Full Model & 5-Check Overfit Test (Twi)

In [ ]:
!test -d /content/kaza && git -C /content/kaza pull --ff-only origin phase1-design || git clone -b phase1-design https://github.com/udohchuks/kaza.git /content/kaza
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
# Build a ~25-30s Twi clip and encode it to [1, T, 12] LiquidCodec tokens
import io, soundfile as sf, torch
from datasets import Audio, load_dataset
from kaza.env import print_env
from kaza.codec import KazaCodec, SAMPLE_RATE, to_mono_24k

env = print_env()
codec = KazaCodec(device=env.device, dtype=env.frozen_dtype)

ds = load_dataset("google/WaxalNLP", "aka_asr", split="train", streaming=True)
ds = ds.cast_column("audio", Audio(decode=False))

chunks, total_samples = [], 0
for row in ds:
    wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))
    w = to_mono_24k(wav, sr)
    chunks.append(w)
    total_samples += len(w)
    if total_samples >= 25 * SAMPLE_RATE:
        break

clip_wav = torch.cat(chunks)[: 30 * SAMPLE_RATE]
tokens = codec.encode(clip_wav)  # [1, T, 12]
print(f"Twi clip: {len(clip_wav) / SAMPLE_RATE:.1f}s -> tokens {tuple(tokens.shape)}")

In [ ]:
# Build KazaModel, run Check #1 (step-0 identity), and snapshot frozen checksum (Check #3)
from kaza.backbone import verify_step0_identity
from kaza.model import KazaModel

model = KazaModel(
    device=env.device,
    frozen_dtype=env.frozen_dtype,
    trainable_dtype=env.trainable_dtype,
    autocast_dtype=env.autocast_dtype,
)
n_train = sum(p.numel() for p in model.trainable_parameters()) / 1e6
print(f"Trainable params: {n_train:.1f}M")

max_diff = verify_step0_identity(model.backbone)
print(f"Check #1 (step-0 identity): max_diff = {max_diff:.6f}")

checksum_before = model.backbone.frozen_checksum()

In [ ]:
# Check #2 (overfit) + Check #3 (frozen weights unchanged)
all_trainable = model.trainable_parameters()
opt = torch.optim.AdamW(all_trainable, lr=3e-4, weight_decay=0.0)

WARMUP_STEPS, MAX_STEPS = 50, 600
model.train()
model.set_warmup(True)

for step in range(1, MAX_STEPS + 1):
    if step == WARMUP_STEPS + 1:
        model.set_warmup(False)
        print(f"--- Step {step}: unfroze 6 new Global GQA layers ---")

    opt.zero_grad(set_to_none=True)
    loss, acc = model.training_step(tokens)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(all_trainable, 1.0)
    opt.step()

    if step % 50 == 0 or step == 1:
        loss_12, acc_12 = model.eval_all_masked(tokens)
        print(f"step {step:3d} | loss={loss.item():.4f} acc={acc.item()*100:5.1f}% | k=12 loss={loss_12:.4f} acc={acc_12*100:5.1f}%")
        if acc_12 >= 0.995 and loss_12 < 0.05:
            print("Converged!")
            break

checksum_after = model.backbone.frozen_checksum()
print(f"Check #3 (frozen checksum): unchanged={checksum_before == checksum_after}")

In [ ]:
# Check #4 (free-running greedy playback from 1s prompt) + Check #5 (ms/step)
import time
from IPython.display import Audio as Play, display

PROMPT_FRAMES = 13  # ~1.04s
num_new = tokens.shape[1] - PROMPT_FRAMES

torch.cuda.synchronize()
t0 = time.perf_counter()
gen_tokens = model.generate(tokens[:, :PROMPT_FRAMES], num_new_frames=num_new, num_passes=3, greedy=True)
torch.cuda.synchronize()
ms_per_step = (time.perf_counter() - t0) * 1000 / num_new

match = (gen_tokens[:, PROMPT_FRAMES:] == tokens[:, PROMPT_FRAMES:]).float().mean().item()
print(f"Check #4 (free-running greedy match): {match*100:.2f}% over {num_new} frames")
print(f"Check #5 (step latency): {ms_per_step:.1f} ms/step (target < 80 ms)")

orig_rec = codec.decode(tokens)[0].numpy()
gen_wav = codec.decode(gen_tokens)[0].numpy()
print("Original (codec reconstructed):"); display(Play(orig_rec, rate=SAMPLE_RATE))
print("Generated (free-running from 1s prompt):"); display(Play(gen_wav, rate=SAMPLE_RATE))